# Predicting Heart-Disease Risk: nested CV, calibration, honest thresholds, fairness
Thin notebook: all logic lives in `src/` (importable, tested). Every step saves its table/figure to `outputs/`.
Equivalent CLI: `python main.py` (add `--fast` for a quick test).

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))      # repo root, so `import src` works from notebooks/
# Kaggle: clone/upload the repo and use  sys.path.insert(0, "/kaggle/working/<repo-name>")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.config import Config
from src import plots
from src.data import load_and_clean
from src.models import build_models
from src.evaluation import split_instability, run_cv, summarize_cv, select_model
from src.calibration import nested_calibration, probability_quality, operating_points
from src.fairness import sex_analysis
from src.interpret import odds_ratios, permutation_table
from src.clustering import run_unsupervised
from src.pipeline import export_model, build_summary

FAST_MODE = False      # True = ~3 min smoke test
cfg = Config(fast=FAST_MODE, target_sensitivity=0.95, cal_method="sigmoid", flip_label="auto")
rng = np.random.default_rng(cfg.seed)
print("Outputs ->", cfg.out)

## 2. Data, cleaning and label-direction check

In [ ]:
ds = load_and_clean(cfg)
X, y, df = ds.X, ds.y, ds.df
y_arr = y.values
plots.plot_eda(df, cfg); plt.show()
display(df.head())

## 3. Why a single 60/20/20 split is unreliable

In [ ]:
demo = split_instability(X, y, cfg)
plots.plot_split_instability(demo, cfg); plt.show()
print({k: f"{np.mean(v):.3f} ± {np.std(v):.3f} (range {min(v):.2f}-{max(v):.2f})" for k, v in demo.items()})

## 4. Nested repeated CV for all models + model selection

In [ ]:
models = build_models(cfg)
cv = run_cv(models, X, y, cfg)
summary, long = summarize_cv(cv, cfg)
display(summary)
plots.plot_cv_boxplots(long, cfg); plt.show()

In [ ]:
sel = select_model(cv, models, cfg)      # corrected paired t-test (Holm) + one-standard-error rule
display(sel["comp"])
name = sel["final"]

## 5. Calibration and a nested operating threshold

In [ ]:
cal = nested_calibration(X, y, models, name, cfg)
prob_q = probability_quality(y_arr, cal, name, cfg); display(prob_q)
op, naive_thr, pred_naive = operating_points(y_arr, cal, cfg, rng); display(op)
plots.plot_calibration_threshold(y_arr, cal, name, cfg); plt.show()

## 6. Performance by sex

In [ ]:
by_sex, gap_df = sex_analysis(X, y_arr, cal, cfg, rng)
display(by_sex); display(gap_df)

## 7. Interpretability

In [ ]:
orr, lr_params = odds_ratios(X, y, models, cfg, rng)
perm = permutation_table(X, y, cal["fold_res"], cfg)
display(orr.round(3))
plots.plot_importance(orr, perm, name, cfg); plt.show()

## 8. Unsupervised analysis (settings chosen without labels)

In [ ]:
unsup = run_unsupervised(X, y, df, cfg)
display(unsup["unsup"]); display(unsup["db"]); display(unsup["prof"])
plots.plot_unsupervised(unsup, y, cfg); plt.show()
plots.plot_dbscan(unsup, cfg); plt.show()

## 9. Export model and final summary

In [ ]:
artifact = export_model(X, y, models, name, cal, ds.info["flipped"], cfg)
R = dict(cfg=cfg, sel=sel, summary=summary, y_arr=y_arr, cal=cal, pred_naive=pred_naive, naive_thr=naive_thr,
         gap_df=gap_df, unsup=unsup, prob_q=prob_q, demo=demo, info=ds.info)
txt = build_summary(R)
(cfg.out / "final_summary.txt").write_text(txt)
print(txt)

## Conclusions & limitations
1. Evaluation matters more than model choice: with about 300 patients, model differences are mostly within fold noise (corrected paired test, Section 4); one 61-patient split can move AUROC by 0.1+.
2. The one-SE rule prefers the simplest adequate model, which is easiest to explain (odds ratios).
3. Check whether recalibration improved Brier score/slope; the nested threshold shows the specificity actually paid for the sensitivity target, and the naive-vs-nested gap measures optimism.
4. Sex-stratified results have wide CIs (about 96 women); the gap test says whether any difference exceeds noise.
5. Unsupervised structure is weak; compare ARI/NMI with 0 (random).
6. Limitations: single-centre, small sample, angiographic disease (not documented MI), no external validation. Future work: validate on other UCI cohorts, richer EHR variables, sex-specific thresholds.